In [1]:
from actual_tests1 import get_testcases
from orig_tests import get_testcases as get_orig_tests
actual_tests = get_testcases()
orig_tests = get_orig_tests()

In [2]:
print(actual_tests[1])

["assert separate_paren_groups('( ) (( )) (( )( ))') == ['()', '(())', '(()())']"]


In [3]:
print(orig_tests[38])

['assert decode_cyclic(encode_cyclic("abc")) == "abc"']


In [4]:
from humaneval_loader import HumanEvalLoader
human_eval = HumanEvalLoader().get_human_eval()

2024-07-31 17:15:51.753556: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-07-31 17:15:51.774178: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-31 17:15:51.774201: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-31 17:15:51.774735: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-31 17:15:51.778592: I tensorflow/core/platform/cpu_feature_guar

In [ ]:
print(human_eval['test'][38]['test'])

In [ ]:
print(human_eval['test'][38]['prompt'])

In [5]:
import json

with open('tests.jsonl', 'r') as json_file:
    json_list = list(json_file)
all_tests = []
for json_str in json_list:
    result = json.loads(json_str)
    all_tests.append(result)

In [6]:
with open('out1.jsonl', 'w') as outfile:
    for idx,entry in enumerate(all_tests):
        entry['given_tests'] = actual_tests[idx]
        json.dump(entry, outfile)
        outfile.write('\n')

In [7]:
with open('out1.jsonl', 'r') as json_file:
    json_list = list(json_file)
all_tests2 = []
for json_str in json_list:
    result = json.loads(json_str)
    all_tests2.append(result)

In [8]:
all_tests[50]['given_tests']

['assert find_closest_elements([1.0, 2.0, 3.0, 4.0, 5.0, 2.2]) == (2.0, 2.2)',
 'assert find_closest_elements([1.0, 2.0, 3.0, 4.0, 5.0, 2.0]) == (2.0, 2.0)']

In [ ]:
## generating mbpp with default testcases
from MBPPLoader import MBPPLoader
mbpp = MBPPLoader()
dataset = mbpp.get_dataset()
prompts = mbpp.get_prompts()
func_names = mbpp.get_func_names()
gen_tests = mbpp.get_generated_testcases()
mbpp_probs = []
for idx,entry in enumerate(dataset):
    # print(entry)
    func_name = func_names[idx]
    # print(func_name)
    tests = '\ndef check(candidate):\n    ' + '\n    '.join(entry['test_list']) + f'\ncheck({func_name})'
    mbpp_probs.append({
        'task_id': "MBPP/" + str(entry['task_id']),
        'prompt': prompts[idx],
        'entry_point': func_name,
        'test': tests,
        'given_tests': [entry['test_list'][0]],
    })
import json
with open('mbpp.jsonl', 'w') as outfile:
    for idx,entry in enumerate(mbpp_probs):
        json.dump(entry, outfile)
        outfile.write('\n')

In [ ]:
prompts = mbpp.get_prompts()

In [ ]:
mbpp_probs[0]

In [ ]:
with open('mbpp.jsonl', 'r') as json_file:
    json_list = list(json_file)
all_tests2 = []
for json_str in json_list:
    result = json.loads(json_str)
    all_tests2.append(result)

In [ ]:
print(all_tests2[1]['test'])

In [ ]:
sum([len(a.split('assert')) -1 for a in test])/len(test)

In [8]:
import json

def write_jsonl(file_path, data):
    with open(file_path, 'w', encoding='utf-8') as file:
        for item in data:
            file.write(json.dumps(item) + '\n')

def read_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            data.append(json.loads(line.strip()))
    return data
human_ldb = []
# Example usage
human = "humaneval-plus.jsonl"
file_path = "humaneval-plus_ldb.jsonl"
human = read_jsonl(human)
print(human[0])
for idx,entry in enumerate(human):
    entry['task_id'] = f'HumanEval/{idx}'
write_jsonl(file_path, human)

mbppplus = "mbpp-plus.jsonl"
file_path = "mbpp-plus_ldb.jsonl"
mbppplus = read_jsonl(mbppplus)
print(mbppplus[0])
for idx,entry in enumerate(mbppplus):
    entry['task_id'] = f'MBPP/{idx}'
write_jsonl(file_path, mbppplus)

{'name': 'HumanEval_HumanEval/0_has_close_elements', 'language': 'py', 'prompt': 'from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    """ Check if in given list of numbers, are any two numbers closer to each other than\n    given threshold.\n    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)\n    False\n    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)\n    True\n    """\n', 'doctests': 'transform', 'prompt_terminology': 'reworded', 'stop_tokens': ['\ndef', '\n#', '\nif', '\nclass'], 'entry_point': 'has_close_elements', 'test': '\n\nimport numpy as np\n\ndef is_floats(x) -> bool:\n    # check if it is float; List[float]; Tuple[float]\n    if isinstance(x, float):\n        return True\n    if isinstance(x, (list, tuple)):\n        return all(isinstance(i, float) for i in x)\n    if isinstance(x, np.ndarray):\n        return x.dtype == np.float64 or x.dtype == np.float32\n    return False\n\n\ndef assertion(out, exp, atol)